# Pipeline Demo - Data Layer (Phase 3)

This notebook demonstrates **only** the data layer: loading SQuAD v2,
printing statistics, showing random samples, and exporting processed data.

It contains **no** retrieval, embeddings, vector database, prompting or LLM
calls - those belong to later phases. The production code lives in `src/`;
this notebook is a demonstration only.

It runs unmodified on Windows, Linux and Google Colab.

In [ ]:
import sys
from pathlib import Path

# Make the project importable from both local and Colab layouts.
project_root = Path().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)

## 1. Load SQuAD v2

The loader downloads the dataset from Hugging Face and caches it automatically.
If the network is unavailable, it falls back to a tiny in-memory sample so the
rest of the demo still runs.

In [ ]:
from src.config import get_settings
from src.data_loader import DataLoader, DatasetLoadingError

settings = get_settings()
loader = DataLoader(settings.dataset, settings.paths)

try:
    loader.load()
    source = 'Hugging Face'
except DatasetLoadingError as exc:
    print('HF load failed, using tiny fallback:', exc)
    loader.load_from_records(
        {
            'train': [
                {
                    'id': 'f1',
                    'title': 'Demo',
                    'context': 'Paris is the capital of France.',
                    'question': 'What is the capital of France?',
                    'answers': {'text': ['Paris'], 'answer_start': [0]},
                },
                {
                    'id': 'f2',
                    'title': 'Demo',
                    'context': 'The sky is blue.',
                    'question': 'What color is the sky?',
                    'answers': {'text': ['blue'], 'answer_start': [8]},
                },
            ],
            'validation': [
                {
                    'id': 'f3',
                    'title': 'Demo',
                    'context': 'Water boils at 100C.',
                    'question': 'Unknown?',
                    'answers': {'text': [], 'answer_start': []},
                }
            ],
        },
        validate=True,
    )
    source = 'fallback'

print('Loaded from:', source)
print('Splits:', loader.splits())
print('Sizes:', loader.split_sizes())

## 2. Dataset statistics

In [ ]:
stats = loader.compute_statistics()
print(stats)
print()
print('Per-split statistics:')
for split, block in loader.statistics().items():
    print(' ', split, block)

## 3. Random samples

In [ ]:
import random

random.seed(42)
train_docs = loader.documents('train')
samples = random.sample(train_docs, k=min(3, len(train_docs)))

for doc in samples:
    print(repr(doc))
    print('  context   :', doc.context)
    print('  question  :', doc.question)
    print('  answers   :', doc.answers)
    print('  has_answer:', doc.metadata['has_answer'])
    print()

## 4. Export processed data

Cleaned documents are written to `data/processed/` in the chosen format.

In [ ]:
for fmt in ('jsonl', 'json', 'csv'):
    path = loader.export('train', fmt=fmt)
    print(fmt, '->', path, '(' + str(path.stat().st_size) + ' bytes)')

---

Next phases will add embeddings, the Chroma vector store, retrieval, prompt
engineering and Groq generation on top of this data layer.